In [ ]:
from google.colab import files
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import zipfile
import io

# ---- Paleta de cores padrão do trabalho (identidade FIAP) ----
from cycler import cycler
ROSA = "#FE0058"           # cor principal / destaque
ROSA_ESCURO = "#A3003A"
ROSA_CLARO = "#FF8AB0"
GRAFITE = "#222222"        # cor secundária
CINZA = "#8C8C8C"
PALETA = [ROSA, GRAFITE, CINZA, ROSA_ESCURO, ROSA_CLARO]
plt.rcParams["axes.prop_cycle"] = cycler(color=PALETA)

print("Arquivo tratado")
upload = files.upload()
arquivos_zip = [nome for nome in upload if nome.lower().endswith(".zip")]
if len(arquivos_zip) != 1:
    raise ValueError("ZIP: olist_bases_tratadas.zip")
caminho_zip = arquivos_zip[0]
with zipfile.ZipFile(caminho_zip) as z:
    nomes = z.namelist()
print(f"ZIP recebido: {caminho_zip} | {len(nomes)} arquivos")
print("\n".join(nomes))

Arquivo tratado


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
with zipfile.ZipFile(caminho_zip) as z:
    esperados = ["olist_orders_tratado.csv", "olist_order_items_tratado.csv"]
    faltantes = [nome for nome in esperados if nome not in z.namelist()]
    if faltantes:
        raise FileNotFoundError(f"O ZIP não contém: {faltantes}. Use a exportação do ETL do grupo.")
    pedidos = pd.read_csv(z.open("olist_orders_tratado.csv"))
    itens = pd.read_csv(z.open("olist_order_items_tratado.csv"))

print("Pedidos:", pedidos.shape, "| Itens:", itens.shape)
display(pedidos.head(3))
display(itens.head(3))

In [ ]:
with zipfile.ZipFile(caminho_zip) as z:
    qualidade = pd.read_csv(z.open('relatorio_qualidade_etl.csv'))
    print('Arquivos de saída no ZIP:', len(z.namelist()))
    print('Relatório de qualidade:')
    display(qualidade)
    print('Registros de pedidos:',len(pedidos),'| Linhas de itens:',len(itens))
    print('Nulos em preço:',itens['preco'].isna().sum())
    print('IDs de pedido duplicados:',pedidos['id_pedido'].duplicated().sum())

In [ ]:
def eda_tabela(df, nome):
    print(f"\n===== {nome} =====")
    print(f"Linhas: {len(df):,} | Colunas: {df.shape[1]}")
    print(f"Duplicadas (linha inteira): {int(df.duplicated().sum()):,}")
    resumo = pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "% nulos": (100*df.isna().mean()).round(2),
        "valores_distintos": df.nunique(dropna=True)
    })
    display(resumo)

eda_tabela(pedidos, "Pedidos tratados")
eda_tabela(itens, "Itens tratados")
print("Status dos pedidos:")
display(pedidos["status_pedido"].value_counts(dropna=False).rename_axis("status").reset_index(name="quantidade"))
print("IDs duplicados em pedidos:", pedidos["id_pedido"].duplicated().sum())
print("IDs de pedidos presentes em itens:", itens["id_pedido"].nunique())

In [ ]:
pedidos_a = pedidos.copy()
itens_a = itens.copy()
pedidos_a["data_compra_pedido"] = pd.to_datetime(pedidos_a["data_compra_pedido"], errors="coerce")
itens_a["preco"] = pd.to_numeric(itens_a["preco"], errors="coerce")

print("Pedidos sem data válida:", pedidos_a["data_compra_pedido"].isna().sum())
print("Itens com preço ausente/não numérico:", itens_a["preco"].isna().sum())
print("Itens com preço <= 0:", itens_a["preco"].le(0).sum())
if pedidos_a["id_pedido"].duplicated().any():
    raise ValueError("Existe ID duplicado em pedidos. Verifique o ETL antes de prosseguir.")

soma_itens = (itens_a.groupby("id_pedido", as_index=False)
                   .agg(valor_produtos=("preco", "sum"),
                        quantidade_itens=("preco", "size")))
entregues = pedidos_a.loc[pedidos_a["status_pedido"].eq("delivered")].copy()
base = entregues.merge(soma_itens, on="id_pedido", how="left", validate="one_to_one")
print("Pedidos entregues:", len(entregues))
print("Pedidos entregues sem itens:", int(base["valor_produtos"].isna().sum()))
base = base.dropna(subset=["data_compra_pedido", "valor_produtos"]).copy()
base["mes"] = base["data_compra_pedido"].dt.to_period("M").dt.to_timestamp()
base["ano"] = base["data_compra_pedido"].dt.year
base["mes_num"] = base["data_compra_pedido"].dt.month
base["dia"] = base["data_compra_pedido"].dt.normalize()
base["dia_semana"] = base["data_compra_pedido"].dt.dayofweek
print("Base final em nível de pedido:", base.shape)
print("Menor e maior data:", base["data_compra_pedido"].min(),base["data_compra_pedido"].max())

In [ ]:
print("Faturamento total do histórico de pedidos entregues: R$",f"{base['valor_produtos'].sum():,.2f}")
print("Pedidos com valor de produtos:",len(base))
descricao = base["valor_produtos"].describe(percentiles=[.25,.5,.75,.9,.99]).rename("Valor R$ por pedido")
display(descricao.to_frame())
q1,q3=base["valor_produtos"].quantile([.25,.75])
iqr=q3-q1
limite=q3+1.5*iqr
outliers=base["valor_produtos"]>limite
print(f"Limite superior do IQR: R$ {limite:,.2f}")
print(f"Pedidos acima do limite: {outliers.sum():,} ({100*outliers.mean():.2f}% dos pedidos)")
print(f"Participação no faturamento: {100*base.loc[outliers,'valor_produtos'].sum()/base['valor_produtos'].sum():.2f}%")
plt.figure(figsize=(10,4.5))
plt.hist(base["valor_produtos"].clip(upper=base["valor_produtos"].quantile(.99)),bins=45)
plt.title("Distribuição do valor dos pedidos (limitada ao P99 para visualização)")
plt.xlabel("Valor do pedido (R$)");plt.ylabel("Número de pedidos")
plt.tight_layout();plt.show()

Indicadores mensais e estatística de estabilidade
O CV = desvio-padrão / média. Como a série de 2017 cresce rapidamente a partir de base baixa, um CV menor em 2018 não comprova por si só causalidade ou previsibilidade futura.

In [ ]:
mensal=(base.groupby("mes",as_index=False)
        .agg(faturamento=("valor_produtos","sum"),
             pedidos=("id_pedido","nunique")))
mensal["ticket_medio"]=mensal["faturamento"]/mensal["pedidos"]
mensal=mensal.sort_values("mes").reset_index(drop=True)
mensal["variacao_mensal_pct"]=mensal["faturamento"].pct_change()*100
serie=mensal[mensal["mes"].between("2017-01-01","2018-07-01")].copy()
serie["variacao_mensal_pct"] = serie["faturamento"].pct_change() * 100
display(serie)
comp=base[base["ano"].isin([2017,2018])&base["mes_num"].between(1,7)]
resumo_anos=(comp.groupby("ano")
             .agg(faturamento=("valor_produtos","sum"),pedidos=("id_pedido","nunique")))
resumo_anos["ticket_medio"]=resumo_anos["faturamento"]/resumo_anos["pedidos"]
print("Comparação justa entre janeiro e julho de 2017 e 2018:")
display(resumo_anos)
for col in ["faturamento","pedidos","ticket_medio"]:
    if 2017 in resumo_anos.index and 2018 in resumo_anos.index:
        print(f"Variação de {col}: {(resumo_anos.loc[2018,col]/resumo_anos.loc[2017,col]-1)*100:.2f}%")
estabilidade=(serie.assign(ano=serie["mes"].dt.year, mes_num=serie["mes"].dt.month)
             .query("mes_num <= 7")
             .groupby("ano")["faturamento"]
             .agg(["count","mean","median","std","min","max"]))
estabilidade["cv_pct"]=100*estabilidade["std"]/estabilidade["mean"]
print("Estatística descritiva dos meses jan-jul:")
display(estabilidade)

In [ ]:
saida=Path("/content/olist_graficos_base_tratada");saida.mkdir(exist_ok=True)
def salvar(nome):
    plt.tight_layout();plt.savefig(saida/nome,dpi=170,bbox_inches="tight");plt.show()

plt.figure(figsize=(12,5))
plt.plot(serie["mes"],serie["faturamento"]/1e6,marker="o")
plt.title("Trajetória mensal do faturamento de produtos entregues")
plt.xlabel("Mês da compra");plt.ylabel("Faturamento (R$ milhões)");plt.grid(alpha=.25)
salvar("01_trajetoria_mensal.png")

c=serie.assign(ano=serie.mes.dt.year,mes_num=serie.mes.dt.month).query("mes_num <= 7")
p=c.pivot(index="mes_num",columns="ano",values="faturamento")/1e6
ax=p.plot(kind="bar",figsize=(11,5))
ax.set_title("Comparativo jan-jul — 2017 x 2018")
ax.set_xlabel("Mês");ax.set_ylabel("Faturamento (R$ milhões)")
ax.set_xticklabels(["Jan","Fev","Mar","Abr","Mai","Jun","Jul"],rotation=0)
salvar("02_comparacao_anual.png")

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(serie["mes"],serie["pedidos"],marker="o")
plt.title("Quantidade mensal de pedidos entregues")
plt.xlabel("Mês da compra");plt.ylabel("Pedidos únicos");plt.grid(alpha=.25)
salvar("03_pedidos_mensais.png")

plt.figure(figsize=(12,5))
plt.plot(serie["mes"],serie["ticket_medio"],marker="o")
plt.title("Ticket médio mensal — valor dos produtos por pedido")
plt.xlabel("Mês da compra");plt.ylabel("Ticket médio (R$)");plt.grid(alpha=.25)
salvar("04_ticket_medio.png")

v=serie.dropna(subset=["variacao_mensal_pct"])
plt.figure(figsize=(12,5))
plt.bar(v["mes"].dt.strftime("%m/%Y"),v["variacao_mensal_pct"])
plt.axhline(0,color="black",linewidth=.8)
plt.xticks(rotation=60);plt.title("Variação mês contra mês")
plt.ylabel("Variação (%)");plt.xlabel("Mês da compra")
salvar("05_variacao_mensal.png")

In [ ]:
tri=mensal[mensal["mes"].between("2017-10-01","2017-12-01")].copy()
print("Faturamento de outubro a dezembro de 2017:")
display(tri)
plt.figure(figsize=(9,5))
plt.bar(tri["mes"].dt.strftime("%m/%Y"),tri["faturamento"]/1000)
plt.title("Concentração mensal de vendas no fim de 2017")
plt.ylabel("Faturamento de produtos (R$ mil)")
salvar("06_out_nov_dez_2017.png")

nov=base[(base["dia"] >= "2017-11-01")&(base["dia"] <= "2017-11-30")]
diario=nov.groupby("dia",as_index=False).agg(faturamento=("valor_produtos","sum"),pedidos=("id_pedido","nunique"))
black_friday=pd.Timestamp("2017-11-24")
fat_bf=diario.loc[diario["dia"].eq(black_friday),"faturamento"].sum()
fat_nov=diario["faturamento"].sum()
sextas=diario[diario["dia"].dt.dayofweek.eq(4)]
media_outras=sextas.loc[sextas["dia"].ne(black_friday),"faturamento"].mean()
print(f"Black Friday 24/11: R$ {fat_bf:,.2f} | {100*fat_bf/fat_nov:.2f}% de novembro | {fat_bf/media_outras:.2f} vezes a média das outras sextas-feiras")
plt.figure(figsize=(12,5))
plt.bar(diario["dia"].dt.day,diario["faturamento"]/1000)
plt.axvline(24,linestyle="--",linewidth=1)
plt.title("Faturamento diário de novembro de 2017 — pico em 24/11")
plt.xlabel("Dia de novembro");plt.ylabel("Faturamento (R$ mil)")
salvar("07_black_friday_diario.png")

In [ ]:
dias_semana=["Seg","Ter","Qua","Qui","Sex","Sáb","Dom"]
linhas=[]
for ano in [2017,2018]:
    inicio=pd.Timestamp(ano,1,1)
    fim=pd.Timestamp(ano,7,31)
    calendario=pd.DataFrame({"dia":pd.date_range(inicio,fim,freq="D")})
    por_dia=base[base["dia"].between(inicio,fim)].groupby("dia")["valor_produtos"].sum().rename("faturamento")
    calendario=calendario.join(por_dia,on="dia").fillna({"faturamento":0})
    calendario["dia_semana"]=calendario["dia"].dt.dayofweek
    media=calendario.groupby("dia_semana")["faturamento"].mean()
    for d,v in media.items():linhas.append({"ano":ano,"dia_semana":d,"faturamento_medio_diario":v})
    uteis=calendario.loc[calendario.dia_semana<=4,"faturamento"].mean()
    fds=calendario.loc[calendario.dia_semana>=5,"faturamento"].mean()
    print(f"{ano}: faturamento médio diário fins de semana = {100*(fds/uteis-1):.1f}% versus dias úteis")
semanal=pd.DataFrame(linhas)
graf=semanal.pivot(index="dia_semana",columns="ano",values="faturamento_medio_diario")
ax=graf.plot(kind="bar",figsize=(11,5))
ax.set_xticklabels(dias_semana,rotation=0)
ax.set_title("Faturamento médio diário por dia da semana — jan-jul")
ax.set_ylabel("Faturamento médio diário (R$)");ax.set_xlabel("Dia da semana")
salvar("08_padrao_semanal.png")

In [ ]:
normal=c.copy()
normal["indice"]=(normal["faturamento"] /
                  normal.groupby("ano")["faturamento"].transform("mean")*100)
ax=normal.pivot(index="mes_num",columns="ano",values="indice").plot(marker="o",figsize=(11,5))
ax.axhline(100,linestyle="--",linewidth=.8)
ax.set_xticks(range(1,8));ax.set_xticklabels(["Jan","Fev","Mar","Abr","Mai","Jun","Jul"])
ax.set_title("Perfil mensal normalizado — média de cada ano = 100")
ax.set_ylabel("Índice da média mensal");ax.set_xlabel("Mês")
ax.grid(alpha=.25)
salvar("09_perfil_mensal_normalizado.png")

In [ ]:
import shutil
mensal.to_csv(saida/"indicadores_mensais.csv",index=False)
resumo_anos.to_csv(saida/"comparativo_jan_jul.csv")
semanal.to_csv(saida/"sazonalidade_semanal.csv",index=False)
diario.to_csv(saida/"novembro_2017_diario.csv",index=False)
arquivo_saida=shutil.make_archive("/content/graficos_e_indicadores_base_tratada","zip",root_dir=saida)
print("Arquivo pronto:",arquivo_saida)
files.download(arquivo_saida)